In [8]:
from huggingface_hub import login
login()

### Find providers and models with tool_choice = 'auto'

In [ ]:
import os, requests

r = requests.get(
    "https://router.huggingface.co/v1/models",
    headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"},
)
for m in r.json()["data"]:
    for p in m.get("providers", []):
        if p.get("supports_tools") and p.get("status") == "live":
            print(f"{m['id']}:{p['provider']}")

## Step 1: Load and prepare the dataset(data source for RAG)

In [13]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_data = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document object
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_data
]

## Step 2: Create the retrieval tool

In [14]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.tools import Tool

bm25_retriever = BM25Retriever.from_documents(docs)

def extract_text(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.invoke(query)
    if results:
        return "\n\n".join([doc.page_content for doc in results[:3]])
    else:
        return "No matching guest information found"

guest_info_tool = Tool(
    name="guest_info_retriever",
    func=extract_text,
    description="Retrieves detailed information about gala guests based on their name or relation.",
)

## Step 3: Integrate the tool with Alfred

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

# Generate the chat interface, including the tools
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3.8-27B:novita",
)
chat = ChatHuggingFace(llm=llm, verbose=True)


tools = [guest_info_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])]
    }

# THe graph
builder = StateGraph(AgentState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

messages = [HumanMessage(content="Tell me about our guest named 'Lady Ada Lovelace'.")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Here's what I found about your guest **Lady Ada Lovelace**:

- **Relation to you:** Best friend
- **Description:** She is an esteemed mathematician, renowned for her pioneering work in mathematics and computing. She is often celebrated as the **first computer programmer** due to her work on Charles Babbage's Analytical Engine.
- **Email:** ada.lovelace@example.com

It sounds like she's a wonderful guest to have — someone with both deep intellectual accomplishments and a close personal bond with you! Let me know if you'd like help with anything else related to the gala.
